# Údaje

## Údaje z Eurostatu {cite}`eurostat`

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# skontroluj, či v pracovnom priečinku existuje podpriečinok data, ak nie vytvor
Path("data").mkdir(exist_ok=True)

def eurostat_to_datetime(s):
    s = s.astype(str)

    if s.str.match(r"^\d{4}$").all():          # ročné: 1995
        return pd.to_datetime(s, format="%Y")

    elif s.str.match(r"^\d{4}-Q[1-4]$").all(): # štvrťročné: 1995-Q1
        return pd.PeriodIndex(s, freq="Q").to_timestamp()

    elif s.str.match(r"^\d{4}-\d{2}$").all():  # mesačné: 1996-01
        return pd.to_datetime(s, format="%Y-%m")

    else:
        return pd.to_datetime(s)

def eurostat_to_parquet(
    url,
    filename,
    index,
    columns,
    values="OBS_VALUE",
    rename=None
):
    df = pd.read_csv(url)

    df[index] = eurostat_to_datetime(df[index]) # prevod do dátumu

    
    df = df.pivot(
        index=index,
        columns=columns,
        values=values
    )

    if rename:
        df = df.rename(columns=rename)

    df.to_parquet(filename)

    return df

def eurostat_single_series_to_parquet(
    url,
    filename,
    name,
    index="TIME_PERIOD"
):
    df = pd.read_csv(url)

    df[index] = eurostat_to_datetime(df[index]) # prevod do dátumu

    df = df[[index, "OBS_VALUE"]]

    df = df.rename(columns={
        "OBS_VALUE": name
    })

    df = df.set_index(index)

    df.to_parquet(filename)

    return df

def eurostat_multiindex_to_parquet(
    url,
    filename,
    columns,
    rename=None
):
    df = pd.read_csv(url)

    df["TIME_PERIOD"] = eurostat_to_datetime(df["TIME_PERIOD"])

    df = df.pivot(
        index="TIME_PERIOD",
        columns=columns,
        values="OBS_VALUE"
    )
    
    df.columns = [
        "_".join(map(str, col))
        for col in df.columns
    ]

    if rename:
        df = df.rename(columns=rename)

    df.to_parquet(filename)

    return df

In [3]:
url_SK_NIPA_Q_1995= (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/"
    "namq_10_gdp/1.0/*.*.*.*.*?"               # namq_10_gdp: NIPA-ESA
    "c[freq]=Q"                                # štvrťročné údaje
    "&c[unit]=CLV20_MEUR"                      # stále ceny; 2020 = 100; milión eur
    "&c[s_adj]=SCA"                            # sezónne a kalendárne očistené 
    "&c[na_item]=B1GQ,P3_S13,P31_S14,P51G,P6,P7" # V tomto poradí: Y,G,C,I,vývoz,dovoz
    "&c[geo]=SK"                               # Slovensko
    "&c[TIME_PERIOD]=ge:1995-Q1"               # prvé pozorovanie
    "&compress=false"                          # nekomprimovať
    "&format=csvdata"                          # csv súbor
    "&formatVersion=2.0"                       # 
    "&lang=en"                                 # anglický jazyk
    "&labels=id")                              # označenie podľa Eurostat ID
url_SK_HDPbc_Q_1995 = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/namq_10_gdp/1.0/*.*.*.*.*" # namq_10_gdp: NIPA-ESA
    "?c[freq]=Q"                               # štvrťročné údaje
    "&c[unit]=CP_MEUR"                         # bežné ceny; milión eur
    "&c[s_adj]=SCA"                            # sezónne a kalendárne očistené 
    "&c[na_item]=B1GQ"                         # HDP
    "&c[geo]=SK"                               # Slovensko
    "&c[TIME_PERIOD]=ge:1995-Q1" 
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)
url_SK_HDPnsa_Q_1995 = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/namq_10_gdp/1.0/*.*.*.*.*"  # namq_10_gdp: NIPA-ESA
    "?c[freq]=Q"                                     # štvrťročné údaje
    "&c[unit]=CLV20_MEUR"                            # stále ceny; 2020 = 100; milión eur
    "&c[s_adj]=NSA"                                  # sezónne neočistené
    "&c[na_item]=B1GQ"                               # HDP
    "&c[geo]=SK"                                     # Slovensko
    "&c[TIME_PERIOD]=ge:1995-Q1"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)
url_SK_HICP_M_1996 = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/prc_hicp_minr/1.0/*.*.*.*" # prc_hicp: prices HICP
    "?c[freq]=M"                                    # mesačné údaje
    "&c[unit]=I25,RCH_A"                            # I25: Index 2025=100, RCH_A: Miera inflácie
    "&c[coicop18]=TOTAL"                            # Všetky ceny
    "&c[geo]=SK"                                    # Slovensko
    "&c[TIME_PERIOD]=ge:1996-01"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)
url_GDP_PPS_A_1995 = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/prc_ppp_ind_1/1.0/*.*.*.*" # prc_ppp_ind_1: prices PPP 
    "?c[freq]=A"                                    # ročné údaje
    "&c[indic_ppp]=VI_PPS_EU27_2020_HAB"            # indexy objemov reálnych výdavkov na osobu
    "&c[ppp_cat18]=GDP"                             # HDP
    "&c[geo]=EU27_2020,DE,AT,SK"                    # Nemecko, Rakúsko, Slovensko
    "&c[TIME_PERIOD]=ge:1995"                       
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

url_LFSI_Q_2003 = (
"https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
"data/dataflow/ESTAT/lfsi_emp_q/1.0/*.*.*.*.*.*.*" # lfsi_emp_q: Labour Force Survey; zamestnanosť; 
                                        # štvrťročne
"?c[freq]=Q"                                       # štvrťročné údaje
"&c[indic_em]=ACT,EMP_LFS" # indic_em: Employment ind; ACT: Persons in the Labour Force; 
                                        # EMP_LFS: Total employment
"&c[s_adj]=SA"                                     # sezónne očistené
"&c[sex]=T"                                        # pohlavie: všetci
"&c[age]=Y15-64"                                   # vekové pásmo
"&c[unit]=THS_PER,PC_POP"                          # tisíc osôb; percento na obyvateľstve
"&c[geo]=SK"
"&c[TIME_PERIOD]=ge:2003-Q1"                          
"&compress=false"
"&format=csvdata"
"&formatVersion=2.0"
"&lang=en"
"&labels=id"
)

url_IRT_ST_M_1990 = (
"https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
"data/dataflow/ESTAT/irt_st_m/1.0/*.*.*?c" # Short-term interest rates mesačne
"[freq]=M"
"&c[int_rt]=IRT_M3,IRT_M12"                # EURIBOR3M; EURIBOR12M
"&c[geo]=EA"                               # Eurozóna
"&c[TIME_PERIOD]=ge:1990-01"
"&compress=false"
"&format=csvdata"
"&formatVersion=2.0"
"&lang=en"
"&labels=id"
)

url_EA_NIPA_Q_1995= (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/"
    "namq_10_gdp/1.0/*.*.*.*.*?"               # namq_10_gdp: NIPA-ESA
    "c[freq]=Q"                                # štvrťročné údaje
    "&c[unit]=CLV20_MEUR"                      # stále ceny; 2020 = 100; milión eur
    "&c[s_adj]=SCA"                            # sezónne a kalendárne očistené 
    "&c[na_item]=B1GQ,P51G"                    # HDP
    "&c[geo]=EA21"                             # Eurozóna
    "&c[TIME_PERIOD]=ge:1995-Q1"               # prvé pozorovanie
    "&compress=false"                          # nekomprimovať
    "&format=csvdata"                          # csv súbor
    "&formatVersion=2.0"                       # 
    "&lang=en"                                 # anglický jazyk
    "&labels=id")                              # označenie podľa Eurostat ID

url_EA_HICP_M_1996 = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/prc_hicp_minr/1.0/*.*.*.*" # prc_hicp: prices HICP
    "?c[freq]=M"                                    # mesačné údaje
    "&c[unit]=I25,RCH_A"                            # I25: Index 2025=100, RCH_A: Miera inflácie
    "&c[coicop18]=TOTAL"                            # Všetky ceny
    "&c[geo]=EA21"                                  # Eurozóna
    "&c[TIME_PERIOD]=ge:1996-01"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

eurostat_to_parquet(
    url_SK_NIPA_Q_1995,
    "data/SK_NIPA_Q_1995.parquet",
    "TIME_PERIOD",
    "na_item",
    rename={
        "B1GQ": "Y",
        "P31_S14": "C",
        "P3_S13": "G",
        "P51G": "I",
        "P6": "V",
        "P7": "D"
    }
)

eurostat_to_parquet(
    url=url_SK_HDPbc_Q_1995,
    filename="Data/SK_HDPbc_Q_1995.parquet",
    index="TIME_PERIOD",
    columns="unit",
    rename={
        "CP_MEUR": "Ycp"
    }
)

eurostat_to_parquet(
    url=url_SK_HDPnsa_Q_1995,
    filename="Data/SK_HDPnsa_Q_1995.parquet",
    index="TIME_PERIOD",
    columns="unit",
    rename={
        "CLV20_MEUR": "Ynsa"
    }
)

eurostat_to_parquet(
    url=url_SK_HICP_M_1996,
    filename="Data/SK_HICP_M_1996.parquet",
    index="TIME_PERIOD",
    columns="unit",
    rename={
        "I25": "HICP",
        "RCH_A": "infl"
    }
)

eurostat_to_parquet(
    url=url_GDP_PPS_A_1995,
    filename="Data/GDP_PPS_A_1995.parquet",
    index="TIME_PERIOD",
    columns="geo",
    rename={
        "EU27_2020": "EU",
        "DE": "DE",
        "AT": "AT",
        "SK": "SK"
    }
)

df = eurostat_multiindex_to_parquet(
    url=url_LFSI_Q_2003,
    filename="Data/LFSI_Q_2003.parquet",
    columns=["indic_em", "unit"],
    rename={
        "ACT_PC_POP": "Miera_participacie",
        "ACT_THS_PER": "Pracovna_sila",
        "EMP_LFS_PC_POP": "Miera_zamestnanosti",
        "EMP_LFS_THS_PER": "Zamestnanost"
    }
)

df["Miera_nezamestnanosti"] = (
    (df["Pracovna_sila"] - df["Zamestnanost"])
    / df["Pracovna_sila"]
    * 100
)

df.to_parquet("Data/LFSI_Q_2003.parquet")

df=eurostat_to_parquet(
    url=url_IRT_ST_M_1990,
    filename="data/EA_IRT_ST_M_1990.parquet",
    index="TIME_PERIOD",
    columns="int_rt",
    rename={
        "IRT_M3": "EURIBOR3M",
        "IRT_M12": "EURIBOR12M",
    }
)

df=eurostat_to_parquet(
    url=url_EA_NIPA_Q_1995,
    filename="data/EA_NIPA_Q_1995.parquet",
    index="TIME_PERIOD",
    columns="na_item",
    rename={"B1GQ": "Y_EA",
           "P51G": "I_EA"
           }
)

df=eurostat_to_parquet(
    url=url_EA_HICP_M_1996,
    filename="Data/EA_HICP_M_1996.parquet",
    index="TIME_PERIOD",
    columns="unit",
    rename={
        "I25": "HICP_EA",
        "RCH_A": "infl_EA"
    }
)
url_SK_GOV10DD_EDPT1_1995= (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/gov_10dd_edpt1/1.0/*.*.*.*.*?c[freq]=A" # Government deficit/surplus, debt
    "&c[unit]=MIO_EUR"                                           # milión eur
    "&c[sector]=S13"                                             # verejná sféra
    "&c[na_item]=B9"                                             # T-G
    "&c[geo]=SK"                                                 # Slovensko
    "&c[TIME_PERIOD]=ge:1995"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

df=eurostat_to_parquet(
    url=url_SK_GOV10DD_EDPT1_1995,
    filename="Data/SK_GOV10DD_EDPT1_1995.parquet",
    index="TIME_PERIOD",
    columns="na_item",
    rename={"B9": "T_G"}
)
url_SK_BOC_C6_Q_1991= (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/bop_c6_q/1.0/*.*.*.*.*.*.*.*?c[freq]=Q" # Balance of payments by country - quarterly
    "&c[currency]=MIO_EUR"                                       # milión eur
    "&c[bop_item]=CA"                                  
    "&c[sector10]=S1"                                            # sektory: všetky sektory
    "&c[sectpart]=S1"                                            # partner: všetky sektory
    "&c[stk_flow]=BAL"                                           # bilancia
    "&c[partner]=WRL_REST"                                       # partner: zvyšok sveta
    "&c[geo]=SK"                                                 
    "&c[TIME_PERIOD]=ge:1991"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

df=eurostat_to_parquet(
    url=url_SK_BOC_C6_Q_1991,
    filename="Data/SK_SK_BOC_C6_Q_1991.parquet",
    index="TIME_PERIOD",
    columns="bop_item",
)

url_NIPA_PC_A_1995=(
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/nama_10_pc/1.0/*.*.*.*?c[freq]=A"
    "&c[unit]=CLV20_EUR_HAB"
    "&c[na_item]=B1GQ"    
    "&c[geo]=EU27_2020,SK"
    "&c[TIME_PERIOD]=ge:1995"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

df=eurostat_to_parquet(
    url=url_NIPA_PC_A_1995,
    filename="Data/NIPA_PC_A_1995.parquet",
    index="TIME_PERIOD",
    columns="geo",
    rename={
        "EU27_2020": "EU",
        "SK": "SK",
    }
)

url_durables_FCS_1995=(
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/namq_10_fcs/1.0/*.*.*.*?c[freq]=Q"
    "&c[unit]=CLV20_MEUR"
    "&c[na_item]=P311_S14"
    "&c[s_adj]=SCA"
    "&c[geo]=SK"
    "&c[TIME_PERIOD]=ge:1995"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

df=eurostat_single_series_to_parquet(
    url=url_durables_FCS_1995,
    filename="Data/durables_FCS_1995.parquet",
    name="C_D",
    index="TIME_PERIOD"
)
url_vacancy_Q_2008=(
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/jvs_q_r21/1.0/*.*.*.*.*.*?" # Job vacancy statistics
    "c[freq]=Q"                                      
    "&c[nace_r2_1]=A-T"                      # úhrnná štatistika
    "&c[sizeclas]=TOTAL"                             # všetky firmy
    "&c[s_adj]=SA"                                   # sezónne očistené
    "&c[indic_em]=JVR"                               # Job Vacancy Rate
    "&c[geo]=SK"                                     
    "&c[TIME_PERIOD]=ge:2008-Q1"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)

df=eurostat_single_series_to_parquet(
    url=url_vacancy_Q_2008,
    filename="Data/vacancy_Q_2008.parquet",
    name="Miera_volnych_miest",
    index="TIME_PERIOD"
)

url_bop_iip6_q=(
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/bop_iip6_q/1.0/*.*.*.*.*.*.*.*?"
    "c[freq]=Q"
    "&c[currency]=MIO_EUR"
    "&c[bop_item]=FA"
    "&c[sector10]=S1"
    "&c[sectpart]=S1"
    "&c[stk_flow]=N_LE"
    "&c[partner]=WRL_REST"
    "&c[geo]=SK"
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id")

df=eurostat_single_series_to_parquet(
    url=url_bop_iip6_q,
    filename="Data/bop_iip6_q.parquet",
    name="NIIP",
    index="TIME_PERIOD"
)


In [17]:
url_SK_BOC_C6_Q_1991= (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
    "data/dataflow/ESTAT/bop_c6_q/1.0/*.*.*.*.*.*.*.*?c[freq]=Q" # Balance of payments by country - 
                                                                 # quarterly
    "&c[currency]=MIO_EUR"                                       # milión eur
    "&c[bop_item]=CA,IN1,D1,D4P__F,IN2"    # v tomto poradí: CA,NFI,NLI(čisté pracovné príjmy),NII,NUT
                                                        
    "&c[sector10]=S1"                                            # sektory: všetky sektory
    "&c[sectpart]=S1"                                            # partner: všetky sektory
    "&c[stk_flow]=BAL"                                           # bilancia
    "&c[partner]=WRL_REST"                                       # partner: zvyšok sveta
    "&c[geo]=SK"                                                 
    "&compress=false"
    "&format=csvdata"
    "&formatVersion=2.0"
    "&lang=en"
    "&labels=id"
)



df=eurostat_to_parquet(
    url=url_SK_BOC_C6_Q_1991,
    filename="Data/SK_SK_BOC_C6_Q_1991.parquet",
    index="TIME_PERIOD",
    columns="bop_item",
    rename={"CA": "CA", 
            "IN1": "NFI",
           "D1": "NLI",
            "D4P__F": "NII",
            "IN2": "NUT"
           }
)

## Údaje z ECB {cite}`ecb`

In [31]:
import pandas as pd

def ecb_to_parquet(
    dataflow,
    series_key,
    colname,
    filename
):

    url = (
        "https://data-api.ecb.europa.eu/service/data/"
        f"{dataflow}/{series_key}"
        "?format=csvdata"
    )

    df = pd.read_csv(url)

    df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"])

    df = (
        df[["TIME_PERIOD", "OBS_VALUE"]]
        .rename(columns={"OBS_VALUE": colname})
        .set_index("TIME_PERIOD")
    )

    df.to_parquet(filename)

    return df


In [32]:
# Cost of borrowing for new short-term loans , Euro area, Monthly
cob = ecb_to_parquet(
    "MIR",
    "M.U2.B.A2J.FM.R.A.2230.EUR.N",
    "CoB",
    "data/CostOfBorrowing.parquet"
)

# M3
M3 = ecb_to_parquet(
    "BSI",
    "M.U2.Y.V.M30.X.1.U2.2300.Z01.E",
    "M3",                              # v mil. €
    "data/M3.parquet"
)

M1 = ecb_to_parquet(
    "BSI",
    "M.U2.Y.V.M10.X.1.U2.2300.Z01.E",
    "M1",
    "data/M1.parquet"
)

## Grafy

### Definície funkcií

#### Všeobecné funkcie a zobrazenie časových radov

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def set_theory_style():

    plt.rcParams.update({
        "figure.figsize": (8,6),
        "figure.dpi": 120,
        "font.size": 12,
        "lines.linewidth": 2,
        "axes.grid": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })


def set_data_style():

    plt.rcParams.update({
        "figure.figsize": (8,4.5),
        "figure.dpi": 120,
        "font.size": 11,
        "lines.linewidth": 2,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
    })

def plot_ts(
    data,
    filename=None,
    labels=None,
    title=None,
    ylabel=None,
    common_sample=True,
    normalize=False,
    zero_line=False,
    recessions=None,
    figsize=None,
    YearLocator=5
):
    """
    Graf priebehu jedného alebo viacerých časových radov.

    Parameters
    ----------
    data : pd.Series, pd.DataFrame alebo list[pd.Series]
        Vstupné časové rady.

    labels : list[str], optional
        Názvy do legendy.

    title : str, optional
        Názov grafu.

    ylabel : str, optional
        Popis osi y.

    common_sample : bool, default=True
        Oreže rady na spoločné obdobie.

    normalize : bool, default=False
        Normalizuje prvé pozorovanie na 100.

    zero_line : bool, default=False
        Nakreslí horizontálnu čiaru v nule.

    recessions : bool, default=False
        Zobrazí vybrané recesie.

    figsize : tuple, optional
        Prepíše predvolenú veľkosť grafu.
    
    YearLocator : interval rokov zobrazovaných na osi x 

    Returns
    -------
    fig, ax, df
    """

    # --------------------------------------------------
    # Vytvorenie DataFrame
    # --------------------------------------------------

    if isinstance(data, pd.Series):
        df = data.to_frame()

    elif isinstance(data, pd.DataFrame):
        df = data.copy()

    elif isinstance(data, (list, tuple)):
        df = pd.concat(data, axis=1)

    else:
        raise TypeError(
            "data musí byť Series, DataFrame alebo list Series"
        )

    # --------------------------------------------------
    # Datetime index
    # --------------------------------------------------

    if not pd.api.types.is_datetime64_any_dtype(df.index):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            raise ValueError(
                "Index nie je datetime a nepodarilo sa ho previesť."
            )

    df = df.sort_index()

    # --------------------------------------------------
    # Legendy
    # --------------------------------------------------

    if labels is not None:

        if len(labels) != len(df.columns):
            raise ValueError(
                "Počet labels sa nerovná počtu radov."
            )

        df.columns = labels

    # --------------------------------------------------
    # Spoločná vzorka
    # --------------------------------------------------

    if common_sample:

        start = max(
            df[c].first_valid_index()
            for c in df.columns
        )

        end = min(
            df[c].last_valid_index()
            for c in df.columns
        )

        df = df.loc[start:end]

    # --------------------------------------------------
    # Normalizácia
    # --------------------------------------------------

    if normalize:

        df = (
            df
            .div(df.iloc[0])
            .mul(100)
        )

    # --------------------------------------------------
    # Graf
    # --------------------------------------------------

    fig, ax = plt.subplots(figsize=figsize)

    for col in df.columns:
        ax.plot(df.index, df[col], label=col)

    # --------------------------------------------------
    # Voľby
    # --------------------------------------------------

    if title:
        ax.set_title(title)

    if ylabel:
        ax.set_ylabel(ylabel)

    if zero_line:
        ax.axhline(
            0,
            color="black",
            lw=0.8
        )

    if recessions:

        for peak, trough in recessions:

            ax.axvspan(
                pd.Timestamp(peak),
                pd.Timestamp(trough),
                color="lightgray",
                alpha=0.3
            )

    # --------------------------------------------------
    # Os x
    # --------------------------------------------------

    
    ax.xaxis.set_major_locator(
        mdates.YearLocator(YearLocator)
    )

    ax.xaxis.set_major_formatter(
        mdates.DateFormatter("%Y")
    )

    fig.autofmt_xdate()

    if len(df.columns) > 1:
        ax.legend()
    
    plt.tight_layout()

    if filename is not None:
        plt.savefig(
            f"Data/{filename}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()

    return fig, ax, df

#### Log-kvadratický trend

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def decompose_log_quadratic(series, log_quadratic=True):
    """
    Rozklad časového radu pomocou kvadratického trendu.

    Parameters
    ----------
    series : pandas.Series

    log_quadratic : bool
        True  -> log-kvadratický trend
        False -> obyčajný kvadratický trend

    Returns
    -------
    out : pandas.DataFrame

        y
        trend
        cycle
        cycle_pct (iba pri logaritmickom trende)

    model : statsmodels RegressionResults
    """

    series = (
        series
        .dropna()
        .astype(float)
        .sort_index()
        .copy()
    )

    t = np.arange(len(series))

    X = pd.DataFrame(
        {
            "const": 1.0,
            "t": t,
            "t2": t**2
        },
        index=series.index
    )

    # premenná použitá v regresii
    if log_quadratic:
        y_reg = np.log(series)
    else:
        y_reg = series

    # odhad trendu
    model = sm.OLS(
        y_reg,
        X
    ).fit()

    trend_reg = pd.Series(
        model.predict(X),
        index=series.index
    )

    # prevod trendu späť do pôvodných jednotiek
    if log_quadratic:

        trend_level = np.exp(trend_reg)

        cycle = (
            np.log(series)
            - trend_reg
        )

        out = pd.DataFrame(
            {
                "y": series,
                "trend": trend_level,
                "cycle": cycle,
                "cycle_pct": 100 * cycle
            },
            index=series.index
        )

    else:

        cycle = series - trend_reg

        out = pd.DataFrame(
            {
                "y": series,
                "trend": trend_reg,
                "cycle": cycle
            },
            index=series.index
        )

    return out, model

#### Cyckické charakteristiky, prevod dataframeu do obrázku

In [4]:
def cycle_characteristics(
    x,
    y,
    name=None,
    max_lag=1,
    acyclic_threshold=0.2
):
    """
    Cyklické charakteristiky premennej x
    vzhľadom na cyklickú zložku HDP y.

    Ak log_quadratic=False:
        cyklická zložka = y - trend
    """

    df = pd.concat([x, y], axis=1).dropna()

    x = df.iloc[:, 0]
    y = df.iloc[:, 1]

    # volatilita
    sd_x = x.std()
    sd_y = y.std()

    rel_vol = sd_x / sd_y

    # súčasná korelácia
    corr0 = x.corr(y)

    # korelácie s posunom
    corrs = {}

    for k in range(-max_lag, max_lag + 1):
        corrs[k] = x.shift(k).corr(y)

    best_lag = max(
        corrs,
        key=lambda k: abs(corrs[k])
    )

    best_corr = corrs[best_lag]

    # cyklickosť
    if abs(best_corr) < acyclic_threshold:
        cyclicality = "acyklická"
        timing = "neurčená"
    
    elif best_corr > 0:
        cyclicality = "procyklická"
    
    else:
        cyclicality = "kontracyklická"
    
    # časová charakteristika
    if cyclicality != "acyklická":
    
        if best_lag < 0:
            timing = "oneskorená"
    
        elif best_lag > 0:
            timing = "predbiehajúca"
    
        else:
            timing = "súbežná"

    out = pd.DataFrame(
        {
            "Premenná": [name],
            "σ": [sd_x],
            "σ/σ(Y)": [rel_vol],
            "corr(Y)": [corr0],
            "Typ": [cyclicality],
            "Posun": [best_lag],
            "corr(max)": [round(float(best_corr), 2)],
            "Vzťah k cyklu": [timing]
        }
    )

    return out, pd.Series(corrs)


def df_to_png0(df, filename, title=None):

    fig, ax = plt.subplots(
        figsize=(12, 2 + 0.4 * len(df))
    )

    ax.axis("off")

    tbl = ax.table(
        cellText=df.map(
            lambda x: f"{x:.2f}".replace(".", ",")
            if isinstance(x, (int, float))
            else x
        ).values,
        rowLabels=df.index,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)

    tbl.auto_set_column_width(
        col=list(range(len(df.columns)))
    )

    tbl.scale(1.1, 1.4)

    if title:
        ax.set_title(title)

    plt.savefig(
        f"Data/{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

### Načítanie dát a definícia recesií 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

SK_RECESSIONS = [
    ("2008-10-01", "2009-01-01"),
    ("2019-10-01", "2020-04-01"),
    ("2021-10-01", "2022-04-01"),
]

EA_RECESSIONS = [
    ("2002-10-01", "2003-01-01"),
    ("2008-01-01", "2009-01-01"),
    ("2011-07-01", "2013-01-01"),
    ("2019-07-01", "2020-04-01"),
    ("2022-07-01", "2023-01-01"),
]

data_dir = Path("Data")

for file in data_dir.glob("*.parquet"):

    var_name = file.stem

    globals()[var_name] = pd.read_parquet(file)

    print(f"Načítané: {var_name}")

Načítané: bop_iip6_q
Načítané: CostOfBorrowing
Načítané: durables_FCS_1995
Načítané: EA_HICP_M_1996
Načítané: EA_IRT_ST_M_1990
Načítané: EA_NIPA_Q_1995
Načítané: GDP_PPS_A_1995
Načítané: LFSI_Q_2003
Načítané: M1
Načítané: M3
Načítané: NAMQ_FCS_1995
Načítané: NIPA_PC_A_1995
Načítané: services_FCS_1995
Načítané: SK_GOV10DD_EDPT1_1995
Načítané: SK_HDPbc_Q_1995
Načítané: SK_HDPnsa_Q_1995
Načítané: SK_HICP_M_1996
Načítané: SK_NIPA_Q_1995
Načítané: SK_SK_BOC_C6_Q_1991
Načítané: vacancy_Q_2008


### Grafy

In [23]:
def graf_Y_Ycp():

    df = pd.concat(
        [
            SK_NIPA_Q_1995["Y"],
            SK_HDPbc_Q_1995["Ycp"]
        ],
        axis=1
    )

    set_data_style()

    fig,ax,df=plot_ts(df,filename="graf_Y_Ycp",
        labels=["Reálny HDP", "Nominálny HDP"],
                      ylabel="v miliónoch €"
    )
    
def graf_Y_Ynsa():
    
    df = pd.concat(
        [
            SK_HDPnsa_Q_1995["Ynsa"],
            SK_NIPA_Q_1995["Y"]
        ],
        axis=1
    )

    set_data_style()

    fig,ax,df=plot_ts(df,filename="graf_Y_Ynsa",
        labels=["Sezónne neočistený reálny HDP","Sezónne a kalendárne očistený reálny HDP"],
                      ylabel="v miliónoch €"
    )
    
def graf_Y():
    df=SK_NIPA_Q_1995["Y"].copy()
    
    set_data_style()

    fig,ax,df=plot_ts(df,filename="graf_Y",
                     recessions=SK_RECESSIONS,
                     ylabel="v miliónoch €")

def graf_lnY():
    df=np.log(SK_NIPA_Q_1995["Y"]).copy()
    
    set_data_style()

    fig,ax,df=plot_ts(df,filename="graf_lnY",
                     recessions=SK_RECESSIONS,
                     ylabel="ln(HDP)")

def graf_HDP_zlozky():
    df = SK_NIPA_Q_1995.copy()

    df["X"] = df["V"] - df["D"]

    fig,ax,df=plot_ts(                      
        df[["Y","C","G","I","X"]],
        filename="graf_HDP_zlozky",
        labels=["Y","C","G","I","X"],
        ylabel="v miliónoch €"
    )

def graf_T_G():
    df=SK_GOV10DD_EDPT1_1995["T_G"].copy()
    fig,ax,df=plot_ts(df,
                     filename="graf_T_G",
                     ylabel="v miliónoch €")

def graf_deflator():
    deflator=100*SK_HDPbc_Q_1995["Ycp"]/SK_NIPA_Q_1995["Y"]
    fig,ax,df=plot_ts(deflator,filename="graf_deflator",recessions=SK_RECESSIONS)
    
def graf_hicp():
    fig,ax,df=plot_ts(SK_HICP_M_1996["HICP"],filename="graf_hicp",recessions=SK_RECESSIONS)
    
def graf_inflacia():
    deflator=100*SK_HDPbc_Q_1995["Ycp"]/SK_NIPA_Q_1995["Y"]
    hicp=SK_HICP_M_1996["HICP"]

    g_deflator = 100 * np.log(deflator).diff(4)
    g_hicp = 100 * np.log(hicp).diff(12)
    g_hicp_q = (g_hicp.resample("QS").mean())

    g_deflator.name = "Deflator"

    df=pd.concat(
        [g_deflator,
         g_hicp
         ],
        axis=1
        )
    df = df.dropna()
    fig,ax,df=plot_ts(df,filename="graf_inflacia",
                      labels=["Inflácia meraná deflátorom HDP", "Inflácia meraná HICP"],
                      recessions=SK_RECESSIONS,
                      ylabel=["%"])
def graf_PPP():
    df=GDP_PPS_A_1995.copy()
    fig,ax,df=plot_ts(df,
                     filename="graf_PPP",
                     recessions=SK_RECESSIONS
                     )

def grafy_Ycz():
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["Y"])
    Ytrend=dec["trend"]

    df=pd.concat(
        [SK_NIPA_Q_1995["Y"].loc["2000":],
         Ytrend.loc["2000":]
        ],
        axis=1
        )

    fig,ax,df=plot_ts(df,
                      filename="graf_Ytrend",
                      labels=["Reálny HDP","Trend reálneho HDP"],
                      recessions=SK_RECESSIONS,
                      ylabel="v miliónoch €"
                    )
    fig,ax,df=plot_ts(dec["cycle_pct"],
                      filename="graf_Ycz",
                      recessions=SK_RECESSIONS,
                      ylabel="%"
                    )

def graf_u():
    fig,ax,df=plot_ts(LFSI_Q_2003["Miera_nezamestnanosti"],
                      filename="graf_u",
                      recessions=SK_RECESSIONS,
                      ylabel="%"
                     )
def graf_q():
    fig,ax,df=plot_ts(LFSI_Q_2003["Miera_participacie"],
                      filename="graf_q",
                      recessions=SK_RECESSIONS,
                      ylabel="%"
                     )
def graf_v():
    fig,ax,df=plot_ts(vacancy_Q_2008,
                      filename="graf_v",
                      recessions=SK_RECESSIONS,
                      ylabel="%"
                     )

def grafy_uv():

    #--------
    # graf_uv
    #--------

    df = pd.concat(
        [
            LFSI_Q_2003["Miera_nezamestnanosti"],
            vacancy_Q_2008["Miera_volnych_miest"]
        ],
        axis=1
    ).loc["2009":].dropna()
    
    df.columns = [
        "Miera nezamestnanosti",
        "Miera voľných miest"
    ]
    
    fig,ax,df=plot_ts(df,
                      filename="graf_uv",
                      labels=["Miera nezamestnanosti","Miera voľných miest"],
                      recessions=SK_RECESSIONS,
                      normalize=True
                     )
    
    #--------------------
    # Beveridgeova krivka
    #--------------------
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    ax.plot(
        df["Miera nezamestnanosti"],
        df["Miera voľných miest"],
        marker="o",
        markersize=3
    )

    # prvé pozorovanie
    first = df.iloc[0]
    
    ax.scatter(
        first["Miera nezamestnanosti"],
        first["Miera voľných miest"],
        color="green",
        s=50,
        zorder=5
    )
    
    ax.annotate(
        df.index[0].strftime("%Y-%m"),
        (
            first["Miera nezamestnanosti"],
            first["Miera voľných miest"]
        ),
        xytext=(7, 7),
        textcoords="offset points"
    )
    
    # posledné pozorovanie
    last = df.iloc[-1]
    
    ax.scatter(
        last["Miera nezamestnanosti"],
        last["Miera voľných miest"],
        color="red",
        s=50,
        zorder=5
    )
    
    ax.annotate(
        df.index[-1].strftime("%Y-%m"),
        (
            last["Miera nezamestnanosti"],
            last["Miera voľných miest"]
        ),
        xytext=(7, 7),
        textcoords="offset points"
    )
    
    ax.set_xlabel("Miera nezamestnanosti (%)")
    ax.set_ylabel("Miera voľných pracovných miest (%)")
    
    plt.tight_layout()
    
    plt.savefig(
        "Data/graf_beveridge.png",
        dpi=300,
        bbox_inches="tight"
    )
    
    plt.close()

#-----------------
# graf_otvorenost
#-----------------

def graf_otvorenost():

    miera_otvorenosti=(SK_NIPA_Q_1995["V"]+SK_NIPA_Q_1995["D"])/SK_NIPA_Q_1995["Y"]*100
    
    fig,ax,df=plot_ts(miera_otvorenosti,
                      filename="graf_otvorenost",
                      recessions=SK_RECESSIONS,
                      ylabel="%"
                     )
    
    X=SK_NIPA_Q_1995["V"]-SK_NIPA_Q_1995["D"]
    
    fig,ax,df=plot_ts(X,
                        filename="graf_X",
                        recessions=SK_RECESSIONS,
                        ylabel="v miliónoch €",
                      zero_line=True
                             )
    
    df = pd.concat(
        [
            SK_SK_BOC_C6_Q_1991[["CA", "NFI", "NUT"]],
            X
        ],
        axis=1
    )
    
    fig,ax,df=plot_ts(df,
                      filename="graf_CAzlozky",
                      recessions=SK_RECESSIONS,
                      labels=["CA","NFI","NUT","X"],
                        ylabel="v miliónoch €",
                      zero_line=True
        )
    
    fig,ax,df=plot_ts(SK_SK_BOC_C6_Q_1991[["NFI", "NLI", "NII"]],
                      filename="graf_NFIzlozky",
                      recessions=SK_RECESSIONS,
                      labels=["NFI", "NLI", "NII"],
                        ylabel="v miliónoch €",
                      zero_line=True
        )
    
    fig,ax,df=plot_ts(bop_iip6_q,
                      filename="graf_NIIP",
                      recessions=SK_RECESSIONS,
                      ylabel="v miliónoch €",
        )

def graf_Mdata():
    df = pd.concat(
        [
            EA_HICP_M_1996["infl_EA"],
            EA_IRT_ST_M_1990["EURIBOR3M"]
        ],
        axis=1
    )
    
    set_data_style()
    
    corr=round(df["infl_EA"].corr(df["EURIBOR3M"]),2)
    
    fig,ax,df_graf=plot_ts(df,
                           filename="graf_fisher",
                        labels=["Miera inflácie", "Nominálna úroková miera"],
                          title = f"Koeficient korelácie: {corr:.2f}",
                            recessions=EA_RECESSIONS,
                          ylabel="%"
    )
    
    r=EA_IRT_ST_M_1990["EURIBOR3M"]-EA_HICP_M_1996["infl_EA"]
    
    fig,ax,df_graf=plot_ts(
        r, 
        filename=("graf_r"),
        recessions=EA_RECESSIONS, ylabel="%",zero_line=True
    )
    
    dfM = pd.concat(
        [
            M1/1000000,
            M3/1000000
        ],
        axis=1
    )
    
    dfM = dfM.loc["1995":]
    
    fig,ax,df_graf=plot_ts(
        dfM,
        filename="graf_M",
        labels=["M1", "M3"],
            recessions=EA_RECESSIONS,
        ylabel="bilión €"
    )

    fig,ax,df=plot_ts(
        EA_NIPA_Q_1995["Y_EA"],
        filename="Graf_Y_EA",
        recessions=EA_RECESSIONS,
        ylabel="v miliónoch €",
    )
    
    fig,ax,df=plot_ts(
        EA_HICP_M_1996["HICP_EA"],
        filename="Graf_P_EA",
        recessions=EA_RECESSIONS,
    )
    
    fig,ax,df=plot_ts(
        EA_HICP_M_1996["infl_EA"],
        filename="Graf_inf_EA",
        recessions=EA_RECESSIONS,
        ylabel="%"
    )

graf_Y_Ynsa()    
graf_Y_Ycp()
graf_Y()
graf_lnY()
graf_HDP_zlozky()
graf_T_G()
graf_deflator()
graf_hicp()
graf_inflacia()
graf_PPP()
grafy_Ycz()
graf_u()
graf_q()
graf_v()
grafy_uv()
graf_otvorenost()
graf_Mdata()

#### Cyklické charakteristiky: zložky HDP a ceny

In [9]:
def cyklicke_charakteristiky():
    
    SK_HICP_Q_1996 = (
        SK_HICP_M_1996
        .resample("QS")
        .mean()
    )

    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["Y"])
    Y_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["C"])
    C_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["G"])
    G_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["I"])
    I_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["V"])
    V_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["D"])
    D_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_HICP_Q_1996["HICP"])
    P_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(SK_HICP_Q_1996["infl"],log_quadratic=False)
    pi_cycle=dec["cycle"].loc["2000":]


    tab_Y, corr_Y = cycle_characteristics(
        Y_cycle,
        Y_cycle,
        name="HDP"
    )

    tab_C, corr_C = cycle_characteristics(
        C_cycle,
        Y_cycle,
        name="Spotreba"
    )

    tab_G, corr_G = cycle_characteristics(
        G_cycle,
        Y_cycle,
        name="Vládne nákupy"
    )

    tab_I, corr_I = cycle_characteristics(
        I_cycle,
        Y_cycle,
        name="Investície"
    )

    tab_V, corr_V = cycle_characteristics(
        V_cycle,
        Y_cycle,
        name="Vývoz"
    )

    tab_D, corr_D = cycle_characteristics(
        D_cycle,
        Y_cycle,
        name="Dovoz",
    )

    tab_P, corr_P = cycle_characteristics(
        P_cycle,
        Y_cycle,
        name="HICP",
    )

    tab_pi, corr_pi = cycle_characteristics(
        pi_cycle,
        Y_cycle,
        name="Inflácia",
    )


    tab=pd.concat(
        [tab_Y,tab_C, tab_I, tab_G, tab_V, tab_D, tab_P
        , tab_pi
        ],
        ignore_index=True
    )

    tab.loc[0, "Typ":"Vzťah k cyklu"] = "---"
    tab.loc[6:7, "Posun":"Vzťah k cyklu"]="---"
    
    df_to_png0(tab.round(2), "tab_cyklicke")

    df_C=pd.concat([Y_cycle,C_cycle],axis=1)
    df_C.columns=["HDP","Spotreba"]
    df_G=pd.concat([Y_cycle,G_cycle],axis=1)
    df_G.columns=["HDP","Vládne nákupy"]
    df_I=pd.concat([Y_cycle,I_cycle],axis=1)
    df_I.columns=["HDP","Investície"]
    df_V=pd.concat([Y_cycle,V_cycle],axis=1)
    df_V.columns=["HDP","Vývoz"]
    df_D=pd.concat([Y_cycle,D_cycle],axis=1)
    df_D.columns=["HDP","Dovoz"]
    df_P=pd.concat([Y_cycle,P_cycle],axis=1)
    df_P.columns=["HDP","Cenová hladina"]
    df_pi=pd.concat([Y_cycle,pi_cycle],axis=1)
    df_pi.columns=["HDP","Inflácia"]

    fig,ax,df=plot_ts(df_C, filename="graf_Ccycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_G, filename="graf_Gcycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_I, filename="graf_Icycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_V, filename="graf_Vcycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_D, filename="graf_Dcycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_P, filename="graf_Pcycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_pi, filename="graf_picycle",
                      recessions=SK_RECESSIONS,ylabel="%")
cyklicke_charakteristiky()

C:\Users\EU\AppData\Local\Temp\ipykernel_20876\4276152558.py:83: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '---' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tab.loc[0, "Typ":"Vzťah k cyklu"] = "---"
C:\Users\EU\AppData\Local\Temp\ipykernel_20876\4276152558.py:83: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '---' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tab.loc[0, "Typ":"Vzťah k cyklu"] = "---"


#### Cyklické charakteristiky spotreby trvalých statkov

In [10]:
def cyklicke_charakteristiky_D():
    
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["Y"])
    Y_cycle=dec["cycle_pct"].loc["2000":]
    dec, model = decompose_log_quadratic(durables_FCS_1995["C_D"])
    C_cycle_D=dec["cycle_pct"].loc["2000":]

    tab_C_D, corr_C_D = cycle_characteristics(
        C_cycle_D,
        Y_cycle,
        name="Spotreba trvalých statkov"
    )
    df_to_png0(tab_C_D.round(2), "tab_C_D")

    df_C_D=pd.concat([Y_cycle,C_cycle_D],axis=1)
    df_C_D.columns=["HDP","Spotreba trvalých statkov"]

    fig,ax,df=plot_ts(df_C_D, filename="graf_C_cycle_D",
                      recessions=SK_RECESSIONS,ylabel="%")
cyklicke_charakteristiky_D()

#### Cyklické charakteristiky na trhu práce

In [5]:
def cyklicke_charakteristiky_praca():
    maxlag=4
    apn=SK_NIPA_Q_1995["Y"]/LFSI_Q_2003["Zamestnanost"]
    dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["Y"])
    Y_cycle=dec["cycle_pct"].loc["2009":]
    dec, model = decompose_log_quadratic(LFSI_Q_2003["Miera_participacie"])
    participacia_cycle=dec["cycle_pct"]
    dec, model = decompose_log_quadratic(LFSI_Q_2003["Miera_zamestnanosti"])
    zamestnanost_cycle=dec["cycle_pct"]
    dec, model = decompose_log_quadratic(LFSI_Q_2003["Miera_nezamestnanosti"])
    nezamestnanost_cycle=dec["cycle_pct"]
    dec, model = decompose_log_quadratic(apn)
    apn_cycle=dec["cycle_pct"]
    dec, model = decompose_log_quadratic(vacancy_Q_2008["Miera_volnych_miest"])
    volne_miesta_cycle=dec["cycle_pct"]

    tab_Y, corr_Y = cycle_characteristics(
        Y_cycle,
        Y_cycle,
        name="HDP",
        max_lag=maxlag
    )
    
    tab_participacia, corr_participacia = cycle_characteristics(
        participacia_cycle,
        Y_cycle,
        name="Miera zapojenia",
        max_lag=maxlag
    )

    tab_zamestnanost, corr_zamestnanost = cycle_characteristics(
        zamestnanost_cycle,
        Y_cycle,
        name="Miera zamestnanosti",
        max_lag=maxlag
    )
    
    tab_nezamestnanost, corr_nezamestnanost = cycle_characteristics(
        nezamestnanost_cycle,
        Y_cycle,
        name="Miera nezamestnanosti",
        max_lag=maxlag
    )

    tab_apn, corr_apn = cycle_characteristics(
        apn_cycle,
        Y_cycle,
        name="Priemerný produkt práce",
        max_lag=maxlag
    )

#    tab_volne_miesta, corr_volne_miesta = cycle_characteristics(
#        volne_miesta_cycle,
#        Y_cycle,
#        name="Miera voľných miest",
#        max_lag=maxlag
#    )

    tab_w = pd.DataFrame({
        "Premenná": ["Mzdová sadzba"],
        "σ": ["---"],
        "σ/σ(Y)": ["---"],
        "corr(Y)": ["---"],
        "Typ": ["procyklická"],
        "Posun": ["---"],
        "corr(max)": ["---"],
        "Vzťah k cyklu": ["---"]
    })

    tab=pd.concat(
        [tab_nezamestnanost,
         tab_participacia,
         tab_zamestnanost,
         # tab_volne_miesta,
         tab_apn,
         tab_w
        ],
        ignore_index=True
    )
    
    df_to_png0(tab.round(2), "tab_praca")

    df_participacia=pd.concat([Y_cycle,participacia_cycle],axis=1)
    df_participacia.columns=["HDP","Miera zapojenia"]
    df_zamestnanost=pd.concat([Y_cycle,zamestnanost_cycle],axis=1)
    df_zamestnanost.columns=["HDP","Miera zamestnanosti"]    
    df_nezamestnanost=pd.concat([Y_cycle,nezamestnanost_cycle],axis=1)
    df_nezamestnanost.columns=["HDP","Miera nezamestnanosti"]
#    df_volne_miesta=pd.concat([Y_cycle,volne_miesta_cycle],axis=1)
#    df_volne_miesta.columns=["HDP","Miera voľných miest"]
    df_apn=pd.concat([Y_cycle,apn_cycle],axis=1)
    df_apn.columns=["HDP","Priemerný produkt práce"]
    
    fig,ax,df=plot_ts(df_participacia, filename="graf_participacia_cycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_zamestnanost, filename="graf_zamestnanost_cycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_nezamestnanost, filename="graf_nezamestnanost_cycle",
                      recessions=SK_RECESSIONS,ylabel="%")
    fig,ax,df=plot_ts(df_apn, filename="graf_apn_cycle",
                      recessions=SK_RECESSIONS,ylabel="%")
#    fig,ax,df=plot_ts(df_volne_miesta, filename="graf_volne_miesta_cycle",
#                      recessions=SK_RECESSIONS,ylabel="%")

# Miera nezamestnanosti je mierne kontracyklická a za hospodárskym cyklom zaostáva. 
# Najsilnejšiu koreláciu s cyklickou zložkou HDP dosahuje pri oneskorení o dva štvrťroky.
#    print(corr_nezamestnanost)
# Miera zapojenia vykazuje slabú procyklickú väzbu. Najvyššiu koreláciu dosahuje pri oneskorení dvoch štvrťrokov.
#    print(corr_participacia)

 cyklicke_charakteristiky_praca()

#### Cyklické charakteristiky bilancií so zvyškom sveta

In [57]:
# Kvôli grafom pracujeme s cyklickými zložkami podielov bilancií na HDP.
# V menovateli je trend HDP (Uribe, Schmitt-Grohéová 2016).
# V tomto kóde sú vypočítané druhé momenty aj pre úrovne CA, X.
# Výsledky sú veľmi podobné (jeseň 2026).

maxlag=0
dec, model = decompose_log_quadratic(SK_NIPA_Q_1995["Y"])
Y_trend=dec["trend"]
Y_cycle=dec["cycle_pct"]

X_Y=(SK_NIPA_Q_1995["V"]-SK_NIPA_Q_1995["D"])/Y_trend*100
CA_Y=SK_SK_BOC_C6_Q_1991["CA"]/Y_trend*100

dec, model = decompose_log_quadratic(X_Y,log_quadratic=False)
X_Y_cycle=dec["cycle"]

dec, model = decompose_log_quadratic(CA_Y,log_quadratic=False)
CA_Y_cycle=dec["cycle"]

tab_X_Y, corr_X_Y = cycle_characteristics(
        X_Y_cycle,
        Y_cycle,
        name="X/Y",
        max_lag=maxlag
    )

tab_CA_Y, corr_CA_Y = cycle_characteristics(
        CA_Y_cycle,
        Y_cycle,
        name="CA/Y",
        max_lag=maxlag
    )

tab_Y=pd.concat(
    [tab_X_Y,
     tab_CA_Y,
    ],
    ignore_index=True
)

tab_Y = tab_Y.drop(tab_Y.columns[-3:], axis=1)

# print(tab_Y)
df_to_png0(tab_Y.round(2), "tab_bilancie")

df_X = pd.concat([Y_cycle,X_Y_cycle],axis=1)
df_X.columns=["HDP","X/Y"]
df_CA=pd.concat([Y_cycle,CA_Y_cycle],axis=1)
df_CA.columns=["HDP","CA/Y"]


fig,ax,df=plot_ts(df_X, filename="graf_Xcycle",
                  recessions=SK_RECESSIONS,ylabel="%")
fig,ax,df=plot_ts(df_CA, filename="graf_CAcycle",
                  recessions=SK_RECESSIONS,ylabel="%")

# Úrovne X,CA

X=SK_NIPA_Q_1995["V"]-SK_NIPA_Q_1995["D"]

dec, model = decompose_log_quadratic(X,log_quadratic=False)
X_cycle=dec["cycle"]

dec, model = decompose_log_quadratic(SK_SK_BOC_C6_Q_1991["CA"],log_quadratic=False)
CA_cycle=dec["cycle"]

tab_X, corr_X_Y = cycle_characteristics(
        X_cycle,
        Y_cycle,
        name="X",
        max_lag=maxlag
    )

tab_CA, corr_CA_Y = cycle_characteristics(
        CA_cycle,
        Y_cycle,
        name="CA",
        max_lag=maxlag
    )

tab=pd.concat(
    [tab_X,
     tab_CA,
    ],
    ignore_index=True
)

# print(tab)

#### Cyklické charakteristiky peňažných ukazovateľov

In [12]:
maxlag=24

# Zatiaľ nepublikujeme
# Cyklický vzťah miery inflácie a úrokovej miery

dec, model = decompose_log_quadratic(EA_HICP_M_1996["infl_EA"],
                                     log_quadratic=False)
iM_cycle=dec["cycle"]
dec, model = decompose_log_quadratic(EA_IRT_ST_M_1990["EURIBOR3M"],
                                     log_quadratic=False)
RM_cycle=dec["cycle"]

tab_i_R, corr_iR = cycle_characteristics(
    iM_cycle,
    RM_cycle,
    max_lag=maxlag
)

# Cyklický vzťah miery rastu M1 a miery inflácie a úrokovej miery

M1_growth = M1.pct_change(12) * 100

tab_Mi, corr_Mi=cycle_characteristics(
    M1_growth,
    EA_HICP_M_1996["infl_EA"],
    max_lag=36
)

EA_HICP_Q_1996 = (
    EA_HICP_M_1996
    .resample("QS")
    .mean()
)

M1Q = (
    M1
    .resample("QS")
    .mean()
)

maxlag=10

dec, model = decompose_log_quadratic(EA_HICP_Q_1996["HICP_EA"])
P_cycle=dec["cycle"]
dec, model = decompose_log_quadratic(EA_HICP_Q_1996["infl_EA"],
                                     log_quadratic=False)
i_cycle=dec["cycle"]
dec, model = decompose_log_quadratic(M1Q["M1"])
M1_cycle=dec["cycle"]
dec, model = decompose_log_quadratic(EA_NIPA_Q_1995["Y_EA"])
Y_cycle=dec["cycle_pct"]

tab_Y, corr_Y=cycle_characteristics(
    Y_cycle,
    Y_cycle,
    name="HDP",
    max_lag=1
)

tab_P, corr_P=cycle_characteristics(
    P_cycle,
    Y_cycle,
    name="Cenová hladina",
    max_lag=maxlag
)

tab_i, corr_i=cycle_characteristics(
    i_cycle,
    Y_cycle,
    name="Miera inflácie",
    max_lag=maxlag
)

tab_M, corr_M=cycle_characteristics(
    M1_cycle,
    Y_cycle,
    name="M1",
    max_lag=maxlag
)

tab=pd.concat(
    [tab_Y,tab_M,tab_P, tab_i],
    ignore_index=True
)

tab.loc[0, "Typ"] = "---"
tab.loc[0, "Vzťah k cyklu"] = "---"

tab.loc[0, "Posun"] = np.nan
tab.loc[0, "corr(max)"] = np.nan

tab.loc[1, "Vzťah k cyklu"] = "---"

tab.loc[1, "Posun"] = np.nan
tab.loc[1, "corr(max)"] = np.nan

df_to_png0(tab.round(2), "tab_cyklickeM")

df_P=pd.concat([Y_cycle,100*P_cycle],axis=1)
df_P.columns=["HDP","Cenová hladina (stonásobok)"]
df_i=pd.concat([Y_cycle,i_cycle],axis=1)
df_i.columns=["HDP","Miera inflácie"]
df_M1=pd.concat([Y_cycle,100*M1_cycle],axis=1)
df_M1.columns=["HDP","M1 (stonásobok)"]


fig,ax,df=plot_ts(df_P, filename="graf_Pcycle_EA",
                  recessions=EA_RECESSIONS,ylabel="%")
fig,ax,df=plot_ts(df_i, filename="graf_i_cycle_EA",
                  recessions=EA_RECESSIONS,ylabel="%")
fig,ax,df=plot_ts(df_M1, filename="graf_M1cycle_EA",
                  recessions=EA_RECESSIONS,ylabel="%")

## Výpočty

In [41]:
import matplotlib.pyplot as plt
import pandas as pd

# výpočet priemerných rastov
def avg_growth(series):

    x0 = series.iloc[0]
    xt = series.iloc[-1]

    n = len(series) - 1

    return (np.log(xt) - np.log(x0)) / n

# dataframe do obrázka
def df_to_png(df, filename, title=None):

    fig, ax = plt.subplots(
        figsize=(6, 2 + 0.4 * len(df))
    )

    ax.axis("off")

    tbl = ax.table(
        cellText=df.map(
            lambda x: f"{x:.2f}".replace(".", ",")
            if isinstance(x, (int, float))
            else x
        ).values,
        rowLabels=df.index,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.1, 1.4)

    if title:
        ax.set_title(title)

    plt.savefig(
        f"Data/{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

### Rast slovenského HDP na obyvateľa a HDP na obyvateľa EÚ v rôznych obdobiach

In [44]:
Y = NIPA_PC_A_1995["SK"]
Ye = NIPA_PC_A_1995["EU"]

g1 = avg_growth(Y.loc["2000":"2008-06"])
g2 = avg_growth(Y.loc["2009-07":"2019-12"])
g3 = avg_growth(Y.loc["2021":])
g = avg_growth(Y.loc["2000":])
h1 = avg_growth(Y.loc["2000":"2009-12"])
h2 = avg_growth(Y.loc["2010":"2019-12"])
h3 = avg_growth(Y.loc["2020":])
ge1 = avg_growth(Ye.loc["2000":"2008-06"])
ge2 = avg_growth(Ye.loc["2009-07":"2019-12"])
ge3 = avg_growth(Ye.loc["2021":])
ge = avg_growth(Ye.loc["2020":])
he1 = avg_growth(Ye.loc["2000":"2009-12"])
he2 = avg_growth(Ye.loc["2010":"2019-12"])
he3 = avg_growth(Ye.loc["2020":])

tab_g = pd.DataFrame(
    {
        "SK": [g1, g2, g3, g],
        "EÚ": [ge1, ge2, ge3, ge]
    },
    index=[
        "2000–2008",
        "2009–2019",
        "2021–2026",
        "Celé obdobie"
    ]
)

tab_h = pd.DataFrame(
    {
        "SK": [h1, h2, h3, g],
        "EÚ": [he1, he2, he3, ge]
    },
    index=[
        "2000–2009",
        "2010–2019",
        "2020–2026",
        "Celé obdobie"
    ]
)

tab_g = (100 * tab_g).round(2)
tab_h = (100 * tab_h).round(2)

df_to_png(
    tab_g,
    "tab_rast_udalosti",
    )

df_to_png(
    tab_h,
    "tab_rast_dekady",
    )

### Rast nominálneho a reálneho HDP

In [45]:
# Rast nminálneho a reálneho HDP

Ycpp=SK_HDPbc_Q_1995["Ycp"].iloc[-1]
Ycp0=SK_HDPbc_Q_1995["Ycp"].loc["2000-01-01"]
Yp=SK_NIPA_Q_1995["Y"].iloc[-1]
Y0=SK_NIPA_Q_1995["Y"].loc["2000-01-01"]

Ycp=SK_HDPbc_Q_1995["Ycp"]
gcp100= 100*avg_growth(Ycp.loc["2000":])*4 #(*4: štvrťročné údaje)
g100=100*g
nasobokcp=Ycpp/Ycp0
nasobok=Yp/Y0

tab_gY= pd.DataFrame(
    {
        "Nominálny HDP": [nasobokcp,gcp100],
        "Reálny HDP": [nasobok,g100]
    },
    index=[
        "Násobok zvýšenia od 2000",
        "Rast v %"
    ]
)

tab_gY = (tab_gY).round(2)


df_to_png(
    tab_gY,
    "tab_Y_Ycp",
    )

### Hospodárske cykly v slovenskej ekonomike a v ekonomike EMÚ

In [51]:
Y = SK_NIPA_Q_1995["Y"].loc["2000":]

troughs = Y[
    (Y < Y.shift(1)) &
    (Y < Y.shift(2)) &
    (Y < Y.shift(-1)) &
    (Y < Y.shift(-2))
]

all_peaks = Y[
    (Y > Y.shift(1)) &
    (Y > Y.shift(2)) &
    (Y > Y.shift(-1)) &
    (Y > Y.shift(-2))
]

# posledný vrchol pred každým dnom
peak_dates = []

for trough_date in troughs.index:

    prev_peaks = all_peaks.loc[all_peaks.index < trough_date]

    if len(prev_peaks) > 0:
        peak_dates.append(prev_peaks.index[-1])

peaks = all_peaks.loc[peak_dates]

In [9]:
Ye = EA_NIPA_Q_1995["Y_EA"]

troughs_e = Ye[
    (Ye < Ye.shift(1)) &
    (Ye < Ye.shift(2)) &
    (Ye < Ye.shift(-1)) &
    (Ye < Ye.shift(-2))
]

all_peaks_e = Ye[
    (Ye > Ye.shift(1)) &
    (Ye > Ye.shift(2)) &
    (Ye > Ye.shift(-1)) &
    (Ye > Ye.shift(-2))
]

# V EA za každým vrcholom nasleduje dno
# posledný vrchol pred každým dnom
# peak_dates_e = []

#for trough_date in troughs_e.index:

#    prev_peaks = all_peaks.loc[all_peaks_e.index < trough_date]

#    if len(prev_peaks) > 0:
#        peak_dates_e.append(prev_peaks.index[-1])

#peaks_e = all_peaks_e.loc[peak_dates_e]

In [10]:
all_peaks_e,troughs_e

(TIME_PERIOD
 2002-10-01    2537287.8
 2008-01-01    2858030.4
 2011-07-01    2807688.9
 2019-07-01    3122288.7
 2022-07-01    3243401.2
 Name: Y_EA, dtype: float64,
 TIME_PERIOD
 2003-01-01    2530424.5
 2009-01-01    2694634.8
 2013-01-01    2758689.2
 2020-04-01    2683246.7
 2023-01-01    3235368.6
 Name: Y_EA, dtype: float64)

In [53]:
peaks,troughs

(TIME_PERIOD
 2008-10-01    19326.4
 2019-10-01    24370.3
 2021-10-01    25117.7
 Name: Y, dtype: float64,
 TIME_PERIOD
 2009-01-01    17443.9
 2020-04-01    21977.3
 2022-04-01    24981.1
 Name: Y, dtype: float64)